To get a working version the code was copied from: https://www.youtube.com/watch?v=AevVRlg6dsc

#### ToDo - links still to visit:
- https://www.gradio.app/guides/creating-a-chatbot-fast
- https://www.gradio.app/guides/chatinterface-examples
- https://medium.com/@vdoddy/lets-build-a-chatbot-a-hands-on-guide-with-langchain-openai-and-gradio-382c2509f2bb
- https://medium.com/@arth2048/gradio-llm-chat-message-history-with-langchain-66c67f0952dc
- https://www.projectpro.io/article/langchain-chatbot/1106?utm_source=chatgpt.com
- https://www.youtube.com/watch?v=7WRKNUXbqEQ

#### Steps to deploy
- create python file that runs locally with the app running on port 80
- create Dockerfile that installs dependencies
- create docker image
- run docker container locally
- push the working docker image to docker hub
- start EC2 instance with ports 80 open AND maybe use userdata?
- pull image (from ECR)
- start docker image on EC2
- test if it works on the public IP

In [2]:
import sys
import os
import openai

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key, tavily_api_key

#openaai.api_key = api_key
os.environ['OPENAI_API_KEY'] = api_key
os.environ['TAVILY_API_KEY'] = tavily_api_key

In [7]:
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_openai import ChatOpenAI
# from langchain_google_genai import ChatGoogleGenerativeAI
# from langchain_anthropic import ChatAnthropic

from langchain.schema import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
import gradio as gr

In [8]:
load_dotenv()

#llm = ChatOpenAI(model="gpt-4o-mini", streaming=True)
#llm = ChatAnthropic(model="claude-3-5-sonnet-20241022",streaming=True)
#llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", streaming=True)
llm = init_chat_model("openai:gpt-4o-mini")

system_message = "you act like an astronaut"

In [12]:
# msg = SystemMessage(system_message)
# llm.invoke(system_message)

High level explenation of the code snippet below

#### How the `stream_response` function works with Gradio

1. **Gradio calls your function automatically**  
   - Every time the user types a message, Gradio calls  
     ```python
     stream_response(message, history)
     ```
   - You don’t call it yourself.  
   - `message` = the latest user input.  
   - `history` = all prior user–AI exchanges as a list of tuples.

2. **Conversation reconstruction**  
   - Inside the function you start with an empty list:  
     ```python
     history_langchain_format = []
     ```
   - Add a system message for context.  
   - Rebuild the past conversation by looping through `history`:  
     ```python
     for human, ai in history:
         history_langchain_format.append(HumanMessage(content=human))
         history_langchain_format.append(AIMessage(content=ai))
     ```
   - This recreates the full dialogue so the model remembers it.

3. **Add the new user input**  
   - Append the current `message` as a `HumanMessage`.

4. **Call the LLM with streaming**  
   - Send the entire `history_langchain_format` list into the LLM.  
   - Use streaming so the output is shown incrementally:
     ```python
     partial_message = ""
     for response in llm.stream(history_langchain_format):
         partial_message += response.content
         yield partial_message
     ```
   - Each `yield` updates the chatbox live with the growing reply.

5. **Why this matters**  
   - Without reconstructing history, the model would only see the
     latest input and forget the conversation.  
   - Streaming provides a better UX by showing the answer as it is generated.


In [ ]:
def stream_response(message, history):
    print(f"Input: {message}. History: {history}\n")

    history_langchain_format = []
    history_langchain_format.append(SystemMessage(content=system_message))

    for human, ai in history:
        #print("There")
        history_langchain_format.append(HumanMessage(content=human))
        history_langchain_format.append(AIMessage(content=ai))

    if message is not None:
        print("Here")
        history_langchain_format.append(HumanMessage(content=message))
        partial_message = ""
        for response in llm.stream(history_langchain_format):
            partial_message += response.content
            yield partial_message


demo_interface = gr.ChatInterface(
    fn=stream_response, 
    textbox=gr.Textbox(
        placeholder="Send to the LLM...",
        container=False,
        autoscroll=True,
        scale=7),
        # type='messages'
)

demo_interface.launch(debug=True, share=True)